In [1]:
from common.mistral import call_mistral, MistralCallConfig
from common.logger import JUPYTER_LOGGER as logger
from common.paths import get_gitignore_data_dpath

import json
from typing import *
import pandas as pd

In [2]:
features_dpath = get_gitignore_data_dpath() / "output"
features_fpaths: list = sorted(features_dpath.glob("*.csv"))
all_features = []
for fpath in features_fpaths:
    df = pd.read_csv(fpath)
    all_features.append(df)
features_df = pd.concat(all_features, ignore_index=True)
features_df = features_df.dropna(subset=["query", "ground_truth"])
features_df = features_df.drop_duplicates(subset=["query"])
logger.info(f"Размер итогового датафрейма признаков: {features_df.shape}")

2026-03-31 23:26:48,076 - jupyter-notebooks - INFO - [JUPYTER 📓] Размер итогового датафрейма признаков: (1391, 1620)


In [3]:
judge_prompt_template = """
Ты — судья, оценивающий точность ответа модели.
Галлюцинация — это когда модель генерирует информацию, не соответствующую фактам и недостоверную.

Вопрос: {query}
Правильный ответ (достоверный эталон): {ground_truth}
Ответ модели: {model_answer}

Определи, является ли ответ модели галлюцинацией.
Выбери строго один вариант и ничего больше: "галлюцинация" или "не галлюцинация".
ТОЛЬКО эти два слова, без кавычек и без дополнительных пояснений.
"""
prompts = [
    judge_prompt_template.format(query=row["query"], ground_truth=row["ground_truth"], model_answer=row["model_answer"])
    for _, row in features_df.iterrows()]
features_df["judge_prompt"] = prompts

C:\Users\User\AppData\Local\Temp\ipykernel_18356\2808530096.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features_df["judge_prompt"] = prompts


In [4]:
from tqdm.auto import tqdm

mistral_cfg = MistralCallConfig(
    models_list=["mistral-small-latest"],
)

judge_system_prompt = (
    "Ты строгий факт-чекер и судья качества ответов. "
    "Проверяй соответствие ответа модели эталонному ответу. "
    "Если доступен внешний инструмент поиска, используй его перед финальным вердиктом. "
    "Отвечай только в требуемом JSON-формате."
)

# Для детерминированной и воспроизводимой оценки.
judge_generation_kwargs = {
    "temperature": 0.1,
    "top_p": 0.9,
    "presence_penalty": 0.0,
    "frequency_penalty": 0.0,
    "reasoning_effort": "high",
    "response_format": {"type": "json_object"},
}

# При наличии интеграции можно передать tools (например, web-search/knowledge-base).
# Если tools пустой, вызов остаётся совместимым с текущим пайплайном.
judge_tools: list[dict[str, Any]] = []
judge_tool_choice: str = "required" if judge_tools else "auto"

output_fpath = get_gitignore_data_dpath() / "judge_scores.csv"
scores = []
queries = []
gt = []
answers = []


def save_scores():
    scores_df = {
        "query": queries,
        "ground_truth": gt,
        "model_answer": answers,
        "judge_score": scores,
    }
    pd.DataFrame(scores_df).to_csv(output_fpath, index=False)


def _normalize_judge_score(raw_response: str) -> str:
    text = raw_response.strip().lower()

    # Предпочитаем структурированный ответ из response_format=json_object.
    try:
        parsed: Any = json.loads(text)
        if isinstance(parsed, dict):
            score_value: Any = parsed.get("judge_score")
            if isinstance(score_value, str):
                normalized = score_value.strip().lower()
                if normalized in {"галлюцинация", "не галлюцинация"}:
                    return normalized
    except json.JSONDecodeError:
        pass

    if "не галлюцинация" in text:
        return "не галлюцинация"
    if "галлюцинация" in text:
        return "галлюцинация"
    return "неизвестно"


def _build_call_kwargs(prompt: str) -> dict[str, Any]:
    messages: list[dict[str, str]] = [
        {"role": "system", "content": judge_system_prompt},
        {"role": "user", "content": prompt},
    ]
    call_kwargs: dict[str, Any] = {
        "messages": messages,
        **judge_generation_kwargs,
    }
    if judge_tools:
        call_kwargs["tools"] = judge_tools
        call_kwargs["tool_choice"] = judge_tool_choice
    return call_kwargs


def make_call(raw):
    prompt = raw.judge_prompt
    call_kwargs = _build_call_kwargs(prompt)
    response = call_mistral(mistral_cfg, **call_kwargs)
    score = _normalize_judge_score(response)
    scores.append(score)
    queries.append(raw.query)
    gt.append(raw.ground_truth)
    answers.append(raw.model_answer)
    save_scores()


# test call
import random

test_raw = features_df.iloc[random.randint(0, features_df.shape[0] - 1)]
make_call(test_raw)
test_score = scores[-1]
logger.info(f"Тестовый вызов завершился. Промпт:\n{test_raw.judge_prompt}\nОценка судьи: {test_score}")

2026-03-31 23:26:51,210 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: начало вызова API
2026-03-31 23:26:51,210 - mistral-call - DEBUG - [MISTRAL 🇫🇷] call_mistral: параметры - ['messages', 'temperature', 'top_p', 'presence_penalty', 'frequency_penalty', 'reasoning_effort', 'response_format']
2026-03-31 23:26:51,216 - mistral-call - INFO - [MISTRAL 🇫🇷] call_mistral: попытка 1/1 с моделью mistral-small-latest, ключ 1/18
2026-03-31 23:26:51,607 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: вызов функции с timeout=240
2026-03-31 23:26:51,609 - mistral-call - DEBUG - [MISTRAL 🇫🇷] safe_call: попытка 1
2026-03-31 23:26:51,611 - mistral-call - DEBUG - [MISTRAL 🇫🇷] sub_call: попытка вызова модели mistral-small-latest
2026-03-31 23:26:52,027 - mistral-call - ERROR - [MISTRAL 🇫🇷] safe_call: ошибка на попытке 1: API error occurred: ...
Traceback (most recent call last):
  File "C:\Users\User\Desktop\dirs\Dev\hack-mfti\common\mistral.py", line 164, in safe_call
    result = future.result(ti

In [15]:
df_already_processed_fpath = get_gitignore_data_dpath() / "judge_scores.csv"
if df_already_processed_fpath.exists():
    existing_df = pd.read_csv(df_already_processed_fpath)
    existing_df["query"] = existing_df["query"].astype("string").str.strip()
    existing_df["model_answer"] = existing_df["model_answer"].astype("string").str.strip()
    existing_df["judge_score"] = existing_df["judge_score"].astype("string").str.strip()
    existing_keys = set(zip(existing_df["query"], existing_df["model_answer"]))
    features_df["query"] = features_df["query"].astype("string").str.strip()
    features_df["model_answer"] = features_df["model_answer"].astype("string").str.strip()
    features_df["already_judged"] = features_df.apply(lambda row: (row["query"], row["model_answer"]) in existing_keys,
                                                      axis=1)
    already_judged_count = features_df["already_judged"].sum()
    total_count = len(features_df)
    logger.info(
        f"Найдено {already_judged_count} из {total_count} ответов, уже оцененных судьей. Эти ответы будут пропущены при повторном запуске.")
    features_df = features_df[~features_df["already_judged"]].copy()
    features_df = features_df.drop(columns=["already_judged"])
    logger.info(f"Размер датафрейма после исключения уже оцененных ответов: {features_df.shape}")
    print(existing_df.head())
else:
    logger.info("Файл с оценками судьи не найден. Все ответы будут оценены.")


2026-03-31 23:25:16,166 - jupyter-notebooks - INFO - [JUPYTER 📓] Найдено 929 из 1685 ответов, уже оцененных судьей. Эти ответы будут пропущены при повторном запуске.
2026-03-31 23:25:16,177 - jupyter-notebooks - INFO - [JUPYTER 📓] Размер датафрейма после исключения уже оцененных ответов: (756, 1623)


                                               query  \
0  Именно эта программа новостей упоминается в пе...   
1  Семитскому божеству МолОх, которому в жертву п...   
2  Этот всем известный арестант был посажен в тюр...   
3  Шахта "Иббенбрюген" недалеко от Рурского бассе...   
4              В финале ЭТОЙ КНИГИ поет Марика Рокк.   

                                   ground_truth  \
0                                   Новости CNN   
1                          Кронос {зачет: Крон}   
2                            "Цыпленок жареный"   
3                                      Антрацит   
4  "Семнадцать мгновений весны" Юлиана Семенова   

                                        model_answer      judge_score  
0  Песня «Навигатор» группы «Кино», написанная Ви...  не галлюцинация  
1                     Ответ на загадку: **Аполлон**.     галлюцинация  
2               Ответ на загадку: **книжный червь**.     галлюцинация  
3  **Шахта «Иббенбрюген» и добыча каменного угля*...     галлюцинаци

In [5]:
pbar = tqdm(total=len(features_df), desc="Оценка ответов судьей")
import logging

logging.disable(logging.DEBUG)
logging.disable(logging.INFO)
for _, row in features_df.iterrows():
    make_call(row)
    pbar.update(1)
pbar.close()
logging.disable(logging.NOTSET)
logger.info(f"Оценка всех ответов завершена. Результаты сохранены в {output_fpath}")

Оценка ответов судьей:   0%|          | 0/1391 [00:00<?, ?it/s]

2026-03-31 23:28:20,369 - mistral-call - ERROR - [MISTRAL 🇫🇷] safe_call: ошибка на попытке 1: API error occurred: ...
Traceback (most recent call last):
  File "C:\Users\User\Desktop\dirs\Dev\hack-mfti\common\mistral.py", line 164, in safe_call
    result = future.result(timeout=timeout)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\User\anaconda3\envs\hack-mfti\Lib\concurrent\futures\_base.py", line 456, in result
    return self.__get_result()
           ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\User\anaconda3\envs\hack-mfti\Lib\concurrent\futures\_base.py", line 401, in __get_result
    raise self._exception
  File "C:\Users\User\anaconda3\envs\hack-mfti\Lib\concurrent\futures\thread.py", line 59, in run
    result = self.fn(*self.args, **self.kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\User\Desktop\dirs\Dev\hack-mfti\common\mistral.py", line 192, in sub_call
    response: Any = client.chat.complete(model=model_name, **sub_kwargs)
             

In [6]:
from __future__ import annotations

import pandas as pd

VALID_LABELS = {"галлюцинация", "не галлюцинация"}
MERGE_KEYS = ["query", "model_answer"]


def normalize_text_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in columns:
        df[col] = df[col].astype("string").str.strip()
    return df


def build_judge_scores_df(
    queries: list[str],
    gt: list[str],
    answers: list[str],
    scores: list[str],
) -> pd.DataFrame:
    df = pd.DataFrame(
        {
            "query": queries,
            "ground_truth": gt,
            "model_answer": answers,
            "judge_score": scores,
        }
    )

    df = normalize_text_columns(df, ["query", "model_answer", "judge_score"])
    df = df[df["judge_score"].isin(VALID_LABELS)].copy()

    # Берем последнюю оценку для одинаковой пары (query, model_answer)
    df = df.drop_duplicates(subset=MERGE_KEYS, keep="last")

    # True = галлюцинация
    df["judge_score"] = df["judge_score"].eq("галлюцинация")
    return df


def attach_judge_scores(
    features_df: pd.DataFrame,
    judge_scores_df: pd.DataFrame,
    existing_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    features_df = normalize_text_columns(features_df, MERGE_KEYS)

    if "judge_score" in features_df.columns:
        features_df = features_df.drop(columns=["judge_score"])

    features_df = features_df.merge(
        judge_scores_df[MERGE_KEYS + ["judge_score"]],
        on=MERGE_KEYS,
        how="left",
    )

    if existing_df is not None:
        existing_df = normalize_text_columns(existing_df, MERGE_KEYS)
        if "judge_score" in existing_df.columns:
            existing_df = existing_df.drop(columns=["judge_score"])

        features_df = pd.concat([features_df, existing_df], ignore_index=True)

    # Убираем дубликаты после объединения источников
    features_df = features_df.drop_duplicates(subset=MERGE_KEYS, keep="last")
    return features_df


# --- main ---
final_scores_df = build_judge_scores_df(queries, gt, answers, scores)
features_df = attach_judge_scores(
    features_df=features_df,
    judge_scores_df=final_scores_df,
    existing_df=locals().get("existing_df"),
)

score_counts = features_df["judge_score"].value_counts(dropna=False)

logger.info("Распределение оценок судьи:")
for score, count in score_counts.items():
    logger.info(f"{score}: {count} ({count / max(len(features_df), 1) * 100:.1f}%)")

matched = int(features_df["judge_score"].notna().sum())
total = len(features_df)
logger.info(
    f"После merge: {matched} из {total} строк имеют оценку судьи "
    f"({matched / max(total, 1) * 100:.1f}%)"
)

output_path = get_gitignore_data_dpath() / "features_with_judge_scores.csv"
features_df.to_csv(output_path, index=False)
logger.info(f"Итоговый датафрейм сохранён: {output_path}")

2026-03-31 23:54:05,312 - jupyter-notebooks - INFO - [JUPYTER 📓] Распределение оценок судьи:
2026-03-31 23:54:05,315 - jupyter-notebooks - INFO - [JUPYTER 📓] True: 1014 (72.9%)
2026-03-31 23:54:05,316 - jupyter-notebooks - INFO - [JUPYTER 📓] False: 367 (26.4%)
2026-03-31 23:54:05,319 - jupyter-notebooks - INFO - [JUPYTER 📓] <NA>: 10 (0.7%)
2026-03-31 23:54:05,321 - jupyter-notebooks - INFO - [JUPYTER 📓] После merge: 1381 из 1391 строк имеют оценку судьи (99.3%)
2026-03-31 23:54:06,969 - jupyter-notebooks - INFO - [JUPYTER 📓] Итоговый датафрейм сохранён: C:\Users\User\Desktop\dirs\Dev\hack-mfti\sber\data\gitignore\features_with_judge_scores.csv
